# DPE aggregation verification with five source seeds

Upload this file as a **new Kaggle notebook**, select **Tesla T4 GPU**, enable **Internet**, then **Run All**. The defaults run both new seeds together in one session; no code edits or attachments are required if the original public version-4 output remains available.

**Exactly two new ResNet-50 backbones are trained: seeds 3 and 4.** The verified seeds 0, 1 and 2 are retrieved and reused. The final primary comparison therefore has **five independent backbone/head seeds**, all on official Waterbirds with the same ResNet-50 architecture, 100-epoch backbone recipe, 15 heads and fixed primary 720/479 validation allocation.

Allow roughly **3–5 hours on T4** for both new backbones plus checks; this is an estimate, not a timed run of this notebook. New backbones train sequentially on the selected GPU. A configurable **8.5-hour worker time limit** leaves time to save partial outputs. Backbone checkpoints are saved after each completed epoch. Preserve the full working directory if resuming; the compact ZIP excludes the large backbone/optimizer states.

The run evaluates all 30 original aggregation configurations independently. It adds matched plain/dropout MLP repetitions and explicit controls to address the remaining questions. It does not search for a test winner or introduce a candidate-selection split, background interventions, another dataset, or another backbone architecture.

**A completed run does not guarantee a positive or statistically decisive result.** Five seeds improve the evidence but can still leave uncertainty; retain every planned seed and regression.


In [ ]:
MODE = "five_seed_verification"
ORIGINAL_SEEDS = [0, 1, 2]             # Reused; no old backbone is retrained.
ADDITIONAL_SEEDS = [3, 4]              # Exactly two NEW backbone/head seeds in this run.
COMBINER_SEEDS = list(range(10000, 10010))

RUN_REPOSITORY_TESTS = True
RUN_COMBINER_STABILITY = True         # 200 fits nested within the five frozen head ensembles.
RUN_CALIBRATION_CONTROLS = True       # Affine calibration and a training-fitted threshold.
RUN_DATA_ALLOCATION_CONTROLS = True   # Uniform heads with all1199 validation examples, two budgets.
RUN_SINGLE_HEAD_CONTROL = True       # Fixed head1, replicated inputs; no test-based head selection.
RUN_PARTITION_SENSITIVITY = True
PARTITION_SEEDS = [1, 2]              # Cheap head/combiner refits on SAME five backbones.
RUN_TEST_BOOTSTRAP = True
RUN_IMAGE_REPLAY = True
BOOTSTRAP_REPLICATES = 10000
BOOTSTRAP_SEED = 20261112
STRICT_RUNTIME = True                # Require verified core versions and a T4.
TIME_BUDGET_HOURS = 8.5

WORK_DIR = "/kaggle/working/dpe_five_seed_verification"
ORIGINAL_OUTPUT_DIR = None           # Optional mounted original version-4 full output.
DATA_DIR = None                      # Optional official Waterbirds directory.
RESUME_FROM = None                   # Optional previous FULL five-seed WORK_DIR under /kaggle/input.
IMAGE_BATCH_SIZE = 256
WORKERS = 2
CPU_THREADS = 2


## What is included

| Check | Scope and purpose |
| --- | --- |
| Source/software/input checks | Pin the already audited implementation and verified numerical/image dependencies; validate source and artifact hashes. |
| Two new backbone/head seeds | Train only3 and4 with the original recipe and epoch-boundary checkpoints. |
| Main aggregation grid | All30 configurations on each of five sources:150 primary configuration/seed results. |
| Matched combiner randomness | Plain/dropout MLP, both weight decays, ten initializations per source:200 fits. These are not200 new backbone seeds. |
| Calibration/threshold controls | Check whether a simple probability/decision-boundary adjustment explains the improvement. Fit only on combiner-training data. |
| Full-validation uniform controls | Use all1199 validation examples for head fitting:20 epochs/40 updates per head, and10 epochs/20 updates. Main heads have20 updates per head. Data pools and selection frequency still differ. |
| Fixed single-head control | Repeat the first head's prediction across input columns, then fit plain/dropout MLPs. Check whether aggregation needs information from different heads. |
| Partition sensitivity | Two additional group-stratified720/479 partitions per backbone, all30 procedures per partition. This adds cheap prototype/combiner fits, **no new backbones**. |
| GPU image replay | Regenerate test predictions from the five final backbone/head checkpoints and compare cached decisions. |
| Uncertainty and final tables | Report original3, new2 and combined5 separately, with paired source-seed intervals, conditional test-example bootstrap and crossed source/test bootstrap. |

The primary result remains the original partition0 experiment. Alternative partitions and combiner initializations are sensitivity analyses nested within five sources; they are never counted as additional independent backbone runs.

Every requested fit, including controls, is frozen before the worker scores test predictions. Backbone feature extraction may create raw test features after the backbone is frozen; those features/test labels are not used to fit heads or combiners.


## Prepare the source and worker dependencies

The pinned source is commit `8e14249f4b3510363be06f627182e2daa9926179`. Core numerical versions and **Pillow13.0.0** match the successful image verification. They are loaded in a fresh subprocess from an isolated package directory, so notebook-kernel imports cannot silently retain another Pillow version. The standard Kaggle PyTorch/CUDA installation is retained; strict checks stop before expensive training if its core versions/hardware differ from the verified environment.

If the public reference notebook now points to a newer experiment, the worker stops. Add the **original version-4 full output** as Kaggle Input and set `ORIGINAL_OUTPUT_DIR` in that case.

To resume an interrupted five-seed run in a new session, attach its **full notebook output**, set `RESUME_FROM` to the prior saved `dpe_five_seed_verification` directory, and restore into the same empty `WORK_DIR`. Keep the recipe unchanged. Completed backbones and frozen fits are reused; partial backbone training resumes from the last completed epoch. The compact ZIP alone cannot resume backbone training.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

if MODE != "five_seed_verification" or ORIGINAL_SEEDS != [0, 1, 2] or ADDITIONAL_SEEDS != [3, 4]:
    raise ValueError("This notebook is scoped to reused0/1/2 and exactly two new seeds3/4.")
if not COMBINER_SEEDS or len(COMBINER_SEEDS) != len(set(COMBINER_SEEDS)):
    raise ValueError("Declare distinct combiner seeds")
if any(type(s) is not int or s < 1 for s in PARTITION_SEEDS) or len(PARTITION_SEEDS) != len(set(PARTITION_SEEDS)):
    raise ValueError("Additional partition seeds must be distinct positive integers; primary partition0 is fixed")
if not 0 < TIME_BUDGET_HOURS <= 10:
    raise ValueError("Choose a positive worker budget no larger than10 hours")
if any(importlib.util.find_spec(m) is None for m in ("torch", "torchvision", "pandas")):
    raise RuntimeError("Use Kaggle's standard Python GPU environment with torch, torchvision and pandas.")

work = Path(WORK_DIR).expanduser().resolve()
if RESUME_FROM:
    prior = Path(RESUME_FROM).resolve()
    if not prior.is_dir():
        raise FileNotFoundError(prior)
    if work.exists() and any(work.iterdir()):
        raise RuntimeError("Restore only into a fresh WORK_DIR. For an already-restored session, set RESUME_FROM=None.")
    shutil.copytree(prior, work)
work.mkdir(parents=True, exist_ok=True)
repo = work / "reference_repo"
commit = "8e14249f4b3510363be06f627182e2daa9926179"
if not (repo / ".git").is_dir():
    repo.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "remote", "add", "origin", "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"], check=True)
try:
    head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    head = None
if head != commit:
    if subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("The private reference checkout was modified. Preserve it and choose a clean working directory.")
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", commit], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)

runtime = work / "runtime_packages"
requirements = ["Pillow==13.0.0", "numpy==2.1.3", "scipy==1.16.3", "PyYAML>=6,<7", "pytest>=8,<10"]
marker = runtime / ".verification_requirements.json"
if not marker.is_file() or json.loads(marker.read_text()) != requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--target", str(runtime), *requirements], check=True)
    marker.write_text(json.dumps(requirements))
config = {name: globals()[name] for name in [
    "MODE", "ORIGINAL_SEEDS", "ADDITIONAL_SEEDS", "COMBINER_SEEDS", "RUN_REPOSITORY_TESTS",
    "RUN_COMBINER_STABILITY", "RUN_CALIBRATION_CONTROLS", "RUN_DATA_ALLOCATION_CONTROLS",
    "RUN_SINGLE_HEAD_CONTROL", "RUN_PARTITION_SENSITIVITY", "PARTITION_SEEDS", "RUN_TEST_BOOTSTRAP",
    "RUN_IMAGE_REPLAY", "BOOTSTRAP_REPLICATES", "BOOTSTRAP_SEED", "STRICT_RUNTIME", "TIME_BUDGET_HOURS",
    "WORK_DIR", "ORIGINAL_OUTPUT_DIR", "DATA_DIR", "IMAGE_BATCH_SIZE", "WORKERS", "CPU_THREADS"]}
config["WORK_DIR"] = str(work)
config_path = work / "config.json"
config_path.write_text(json.dumps(config, indent=2))
print("Prepared source commit:", commit)
print("Reuse existing backbones:", ORIGINAL_SEEDS)
print("Train exactly two new backbones:", ADDITIONAL_SEEDS)
print("Primary total:", len(ORIGINAL_SEEDS) + len(ADDITIONAL_SEEDS), "source seeds")
print("Working directory:", work)


## Embedded verification worker

This contains the complete orchestration, controls, caching, numerical checks and uncertainty calculations. It calls the preserved low-level training functions directly for source seeds3/4. A narrow in-memory adapter broadens the stage-0 configuration factory's0/1/2 seed guard to0–4: the original factory supplies every field, then only the seed is set to3 or4. The legacy study wrapper is bypassed. Source files, loss, augmentation and all other training settings remain unchanged; the adapter is recorded in execution_contract.json.

Input identities and the worker hash protect cached fits. Changing an input or fit recipe requires a distinct result context; it must not silently replace an old fit. Results/ZIPs are refreshed as source seeds freeze and finish. Missing/interrupted seeds cannot be labeled a completed five-seed result.


In [ ]:
RUNNER_SOURCE = r'''"""Kaggle-side verification; this file is embedded in the uploadable notebook."""
import csv, hashlib, json, os, platform, shutil, subprocess, sys, time, urllib.parse, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

C = json.loads(Path(sys.argv[1]).read_text())
W = Path(C['WORK_DIR']); REPO = W / 'reference_repo'
RUN_TAG = C['MODE'] + '-' + hashlib.sha256((json.dumps(C, sort_keys=True)+hashlib.sha256(Path(__file__).read_bytes()).hexdigest()).encode()).hexdigest()[:10]
OUT = W / 'results' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
import numpy as np
import scipy
import torch
import PIL
from src.methods.dpe_head_reweighting import aggregation_source as sources
from src.methods.dpe_head_reweighting import ensemble, neural_combiner as neural, weighting_rules as rules
from src.methods.dpe_head_reweighting import stage0
from src.methods.dpe_head_reweighting.rule_suite import notebook_code_hash
from src.datasets.waterbirds import Waterbirds
from src.models.resnet import load_resnet_checkpoint
from torch.utils.data import DataLoader

COMMIT = '8e14249f4b3510363be06f627182e2daa9926179'
SCRIPT_VERSION = '367976457'
STUDY_ID = 'aggregation-standard-seeds012-0dcf563966c5'
GROUPS = ['00', '01', '10', '11']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(C['CPU_THREADS'])


def log(message):
    print(time.strftime('%H:%M:%S'), message, flush=True)


def write(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')
    temporary.replace(path)


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()


def table(path, rows):
    if not rows: return
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)


def archive():
    """Compact reports/models, not images, large features, or backbone checkpoints."""
    target = W / 'verification_results.zip'; temporary = W / 'verification_results.zip.tmp'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as z:
        for root in [W / 'results', W / 'logs']:
            if root.exists():
                for p in root.rglob('*'):
                    if p.is_file() and p.suffix != '.npy':
                        z.write(p, p.relative_to(W).as_posix())
        z.write(W / 'config.json', 'config.json')
    temporary.replace(target)
    return target


def download(url, path):
    path = Path(path)
    if path.is_file(): return
    path.parent.mkdir(parents=True, exist_ok=True); part = path.with_suffix(path.suffix + '.part')
    with urllib.request.urlopen(url, timeout=120) as response, part.open('wb') as stream:
        shutil.copyfileobj(response, stream, length=1024 * 1024)
    part.replace(path)


def get_reference():
    evidence = W / 'original_evidence' / 'dpe_head_aggregation_session'
    study = evidence / 'studies' / STUDY_ID
    # A mounted notebook output may contain version 4's complete directory.
    mounted = C.get('ORIGINAL_OUTPUT_DIR')
    if not mounted:
        for root in Path('/kaggle/input').rglob(STUDY_ID):
            if root.is_dir() and (root / 'plan.json').is_file():
                mounted = str(root.parent.parent); break
    if mounted:
        mounted = Path(mounted)
        if mounted.name != 'dpe_head_aggregation_session':
            mounted = mounted / 'dpe_head_aggregation_session'
        if not (mounted / 'studies' / STUDY_ID / 'plan.json').is_file():
            raise ValueError('ORIGINAL_OUTPUT_DIR must contain the version-4 study directory.')
    index, token, page = [], None, 0
    if not mounted:
        # Public endpoint: never require or print Kaggle credentials or signed URLs.
        while True:
            args = {'userName': 'noerula', 'kernelSlug': 'dpe-reweighting', 'pageSize': 500}
            if token: args['pageToken'] = token
            url = 'https://www.kaggle.com/api/v1/kernels/output?' + urllib.parse.urlencode(args)
            with urllib.request.urlopen(url, timeout=60) as response: body = json.load(response)
            files = body.get('files', [])
            if any('/kf/' + SCRIPT_VERSION + '/' not in item['url'] for item in files):
                raise RuntimeError('The public notebook now points to another version. Add the ORIGINAL version-4 notebook output as Kaggle Input and set ORIGINAL_OUTPUT_DIR. Never substitute a later version.')
            index.extend(files); page += 1
            if page % 5 == 0: log(f'Listed {page} reference-output pages')
            token = body.get('nextPageToken')
            if not token: break
        lookup = {item['fileName']: item['url'] for item in index}
    else:
        lookup = {'dpe_head_aggregation_session/' + p.relative_to(mounted).as_posix(): p
                  for p in mounted.rglob('*') if p.is_file()}
    wanted = []
    for name, location in lookup.items():
        if '/repository-' in name: continue
        take = ('/studies/' + STUDY_ID + '/' in name and name.endswith(('/plan.json', '/models.json', '/metrics.json', '/summary.json')))
        take |= ('/head_sources/' in name and name.endswith(('/ensemble.pt', '/manifest.json', '/heads.json', '/combiner_training.npz', '/test.npz')))
        take |= ('/features/' in name and name.endswith(('/metadata.csv', '/handoff.json', '/provenance.json')))
        if C['RUN_IMAGE_REPLAY'] or C['RUN_DATA_ALLOCATION_CONTROLS'] or C['RUN_PARTITION_SENSITIVITY']:
            take |= ('/features/' in name and name.endswith(('/val_raw.npz', '/test_raw.npz')))
        if C['RUN_IMAGE_REPLAY']:
            take |= ('/backbones/' in name and name.endswith('/ckpt_last.pt'))
        if take: wanted.append((name, location))
    def fetch(item):
        name, location = item; target = evidence.parent / name
        if mounted:
            if not target.exists(): target.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(location, target)
        else: download(location, target)
        return name
    for _ in ThreadPoolExecutor(max_workers=4).map(fetch, wanted): pass
    if not (study / 'plan.json').is_file(): raise RuntimeError('Missing immutable version-4 plan.json.')
    plan = json.loads((study / 'plan.json').read_text())
    for name, expected in plan['implementation'].items():
        actual = notebook_code_hash(REPO / name) if name.endswith('.ipynb') else sha(REPO / name)
        if actual != expected: raise RuntimeError('Pinned implementation hash mismatch: ' + name)
    write(OUT / 'reference_plan.json', plan)
    write(OUT / 'reference_identity.json', dict(kaggle_version=4, script_version=SCRIPT_VERSION, study_id=STUDY_ID, source_commit=COMMIT))
    return evidence, study, plan


def source_data(source):
    manifest = json.loads((source / 'experiment/manifest.json').read_text())
    heads = json.loads((source / 'experiment/heads.json').read_text())
    if sha(source / 'checkpoints/ensemble.pt') != heads['lock']['ensemble']: raise ValueError('Head hash mismatch')
    if sha(source / 'experiment/manifest.json') != heads['lock']['manifest']: raise ValueError('Manifest hash mismatch')
    if manifest['parts'] != sources.partition(manifest['validation']): raise ValueError('Partition mismatch')
    parts = manifest['parts']; a = {r['id'] for r in parts['head_training']}; b = {r['id'] for r in parts['combiner_training']}
    if len(a) != 720 or len(b) != 479 or a & b: raise ValueError('Invalid 720/479 separation')
    p, y, g, ids = sources.load_cache(source, 'combiner_training', manifest, heads['head_ids'], heads['lock']['ensemble'])
    return manifest, heads, (p, y, g, ids)


def metrics(p, y, groups):
    # Independent count calculation, including argmax ties going to class 0.
    correct = np.asarray(p).argmax(-1) == y
    counts = {g: int(np.sum(groups == g)) for g in GROUPS}
    if min(counts.values()) == 0: raise ValueError('Missing group')
    accuracy = {g: float(correct[groups == g].mean()) for g in GROUPS}
    return {'wga': min(accuracy.values()), 'overall': float(correct.mean()), 'groups': accuracy, 'counts': counts}


def numpy_predict(row, p):
    """Independent forward calculation: does not call the model's PyTorch forward."""
    p = np.asarray(p, dtype=np.float64)
    if row['family'] == 'uniform': return np.einsum('h,hnc->nc', np.ones(len(p)) / len(p), p)
    if 'weights' in row: return np.einsum('h,hnc->nc', np.asarray(row['weights']), p)
    model = row['model']; state = {k: np.asarray(v) for k, v in model['state'].items()}
    q = p[:, :, 1].T; x = 2 * q - 1
    if model['kind'] == 'logistic': z = x @ state['network.weight'].T + state['network.bias']
    else: z = np.tanh(x @ state['network.0.weight'].T + state['network.0.bias']) @ state['network.2.weight'].T + state['network.2.bias']
    if model['kind'] == 'gate':
        weight = np.exp(z - z.max(1, keepdims=True)); weight /= weight.sum(1, keepdims=True)
        value = (weight * q).sum(1)
    else: value = 1 / (1 + np.exp(-z[:, 0]))
    return np.column_stack([1 - value, value])


def fit_configurations(p, y, g, config, source_seed):
    # Same declared 30 procedures; direct fit supports new source seeds without
    # changing the preserved library's legacy 0/1/2 study-wrapper restriction.
    rows = [dict(id='uniform', family='uniform', parameters={}, model=None, fit=None)]
    for family in neural.FAMILIES:
        dropout = config['prediction_dropout'] if family.endswith('_dropout') else 0.
        for wd in config['weight_decays']:
            model, fitted = neural.fit(p, y, g, config, family.split('_')[0], wd, dropout,
                                       source_seed + config['seed_offset'])
            rows.append(dict(id=f'{family}-wd{wd}', family=family,
                             parameters=dict(weight_decay=wd, prediction_dropout=dropout), model=model, fit=fitted))
    rows += [r for r in rules.fit_all_rules(p, y, g, config['weighting_rules']) if r['id'] != 'uniform']
    if len(rows) != 30 or len({r['id'] for r in rows}) != 30: raise ValueError('Incomplete configuration grid')
    return rows


def get_dataset():
    explicit = C.get('DATA_DIR')
    if explicit:
        if not Path(explicit).is_dir(): raise FileNotFoundError(explicit)
        return Path(explicit)
    # Official script downloads once, verifies official counts, and prepares metadata.
    target = W / 'data'
    if not (target / 'waterbirds/metadata_waterbirds.csv').is_file():
        subprocess.run([sys.executable, 'scripts/prepare_waterbirds.py', '--data_dir', str(target)], check=True)
    return target / 'waterbirds'


def image_replay(seed, source, checkpoint, feature_directory, dataset, frozen_rows, reference_p, y, groups):
    if DEVICE != 'cuda': raise RuntimeError('Enable a Kaggle GPU for image replay, or set RUN_IMAGE_REPLAY=False for cached verification only.')
    manifest = json.loads((source / 'experiment/manifest.json').read_text())
    if sha(checkpoint) != manifest['backbone_hash']: raise ValueError('Backbone checkpoint hash differs from source manifest')
    handoff = json.loads((feature_directory / 'handoff.json').read_text())
    if sha(feature_directory / 'test_raw.npz') != handoff['artifacts']['test_raw.npz']: raise ValueError('Raw test feature hash mismatch')
    head_record = ensemble.load(source / 'checkpoints/ensemble.pt', 2048, 15)
    model = load_resnet_checkpoint(checkpoint, device=DEVICE); model.eval()
    ds = Waterbirds(dataset, 'test')
    with np.load(feature_directory / 'test_raw.npz', allow_pickle=False) as cache:
        old_raw = cache['features']; expected_ids = cache['ids'].astype(str)
    if ds.filenames != expected_ids.tolist(): raise ValueError('Image/feature ID order differs')
    loader = DataLoader(ds, batch_size=C['IMAGE_BATCH_SIZE'], shuffle=False, num_workers=C['WORKERS'])
    chunks = []
    with torch.inference_mode():
        for index, (_, inputs, _, _) in enumerate(loader):
            chunks.append(model.extract_features(inputs.to(DEVICE)).cpu().numpy())
            if index % 10 == 0: log(f'GPU image replay seed {seed}: {sum(len(x) for x in chunks)}/{len(ds)}')
    raw = np.concatenate(chunks)
    p = ensemble.predict(head_record, raw, batch_size=C['IMAGE_BATCH_SIZE'], device=DEVICE)
    report = dict(source_seed=seed,images=len(ds),pillow=PIL.__version__,
                  feature_max_abs_diff=float(np.abs(raw-old_raw).max()),
                  feature_relative_l2=float(np.linalg.norm(raw-old_raw)/np.linalg.norm(old_raw)),
                  head_probability_max_abs_diff=float(np.abs(p-reference_p).max()),
                  changed_head_decisions=int((p.argmax(2)!=reference_p.argmax(2)).sum()), configurations={})
    for row in frozen_rows:
        old=numpy_predict(row,reference_p); fresh=numpy_predict(row,p)
        report['configurations'][row['id']]=dict(metrics=metrics(fresh,y,groups),
                changed_decisions=int((old.argmax(1)!=fresh.argmax(1)).sum()),
                probability_max_abs_diff=float(np.abs(old-fresh).max()))
    write(OUT / f'images/seed{seed}.json', report)
    del model; torch.cuda.empty_cache()
    log(f'Finished GPU image replay seed {seed}; results recorded for all 30 configurations')
    return report


def bootstrap(seed_records, names):
    if not seed_records: return
    correctness=[]; expected=None
    for rec in seed_records:
        p,y,g,ids=rec['test']; rows={r['id']:r for r in rec['original_rows']}
        if expected is not None and not np.array_equal(ids,expected): raise ValueError('Cross-seed test IDs differ')
        expected=ids
        correctness.append([numpy_predict(rows[name],p).argmax(1)==y for name in names])
    a=np.asarray(correctness); ns=len(a); rng=np.random.default_rng(C['BOOTSTRAP_SEED']); draws=[]
    for group in GROUPS:
        mask=g==group; patterns,counts=np.unique(a[:,:,mask].reshape(ns*len(names),-1).T,axis=0,return_counts=True)
        sampled=rng.multinomial(int(mask.sum()),counts/counts.sum(),size=C['BOOTSTRAP_REPLICATES'])
        draws.append((sampled@patterns.astype(float)/mask.sum()).reshape(C['BOOTSTRAP_REPLICATES'],ns,len(names)))
    b=np.min(draws,axis=0); point=np.min([a[:,:,g==group].mean(2) for group in GROUPS],axis=0); results=[]
    for wd in [.01,.001]:
        method=f'mlp_dropout-wd{wd}'
        for reference in ['uniform',f'mlp-wd{wd}']:
            i,j=names.index(method),names.index(reference); delta=100*(b[:,:,i]-b[:,:,j]); observed=100*(point[:,i]-point[:,j])
            lo,hi=np.percentile(delta.mean(1),[2.5,97.5]); results.append(dict(method=method,reference=reference,
                mean_delta_pp=float(observed.mean()),per_seed_delta_pp=observed.tolist(),conditional_95_percentile_pp=[float(lo),float(hi)]))
    write(OUT/'paired_bootstrap.json',dict(replicates=C['BOOTSTRAP_REPLICATES'],seed=C['BOOTSTRAP_SEED'],
        interpretation='Pointwise paired group-stratified test-example intervals conditional on these fitted models; not uncertainty across newly trained backbones. Same resamples across source seeds; minimum group recomputed each replicate.',results=results))


def contracts():
    # Independent, small numerical contracts. No test-label fitting.
    q=torch.tensor([.1,.2,.4,.6,.7,.8,.9,.6,.8,.7],dtype=torch.float64,requires_grad=True)
    y=torch.tensor([0.,0.,0.,1.,1.,1.,1.,1.,1.,1.],dtype=torch.float64)
    g=torch.tensor([0,1,1,2,2,2,3,3,3,3]); raw=q.detach().numpy(); labels=y.numpy()
    loss=-labels*np.log(raw)-(1-labels)*np.log(1-raw)
    expected=np.mean([loss[g.numpy()==i].mean() for i in range(4)])
    np.testing.assert_allclose(float(neural.group_loss(q,y,g).detach()),expected,atol=1e-14)
    keep=neural.prediction_mask(torch.ones(100,15),1.,torch.Generator().manual_seed(0))
    assert bool((keep.sum(1)==1).all())
    with neural.cpu_randomness(0): model=neural.Combiner('mlp',15,8)
    inputs=torch.full((2,15),.8,dtype=torch.float64); mask=torch.ones_like(inputs,dtype=torch.bool);mask[:,0]=False
    captured=[];hook=model.network[0].register_forward_pre_hook(lambda m,args:captured.append(args[0].detach().numpy()))
    model(inputs,mask);hook.remove();np.testing.assert_allclose(captured[0][:,0],0);np.testing.assert_allclose(captured[0][:,1:],.6)
    write(OUT/'numerical_contracts.json',dict(group_balanced_loss='passed',all_dropped_fallback='passed',neutral_prediction_masking='passed'))


# Five-source verification orchestration; no preserved method implementation is patched.
import signal
from scipy.special import expit
from scipy.stats import t as student_t
from src.methods.dpe_head_reweighting import train_heads


# The preserved stage-0 API also hard-codes seeds0/1/2. Broaden only that guard
# in memory; build every other config field using the unchanged original factory.
ORIGINAL_BACKBONE_CONFIG = stage0.backbone_config


def extended_backbone_config(data_dir,output_dir,track='standard',seed=0,device='cpu',workers=4):
    if type(seed) is not int or seed not in (0,1,2,3,4):raise ValueError('Declared source seeds are0 through4 only')
    config=ORIGINAL_BACKBONE_CONFIG(data_dir,output_dir,track,seed if seed<3 else 0,device,workers)
    if seed>=3:config.training.seed=seed
    return config.validate()


stage0.backbone_config = extended_backbone_config


def fingerprint(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()


def memo(name, identity, compute):
    path=OUT/'frozen'/name
    if path.exists():
        saved=json.loads(path.read_text())
        if saved['identity']!=identity: raise ValueError('Frozen fit inputs changed: '+name)
        return saved['value']
    value=compute();write(path,dict(identity=identity,value=value));return value


def partition_rows(rows, seed):
    rng=np.random.RandomState(seed);parts={'head_training':[],'combiner_training':[]}
    for group in GROUPS:
        members=sorted([r for r in rows if r['group']==group],key=lambda r:r['id'])
        quotas=np.array([.6,.4])*len(members);counts=np.floor(quotas).astype(int)
        for j in np.argsort(-(quotas-counts),kind='stable')[:len(members)-counts.sum()]:counts[j]+=1
        order=rng.permutation(len(members));offset=0
        for role,count in zip(parts,counts):
            parts[role].extend(members[i] for i in order[offset:offset+count]);offset+=count
    for role in parts:
        ids={r['id'] for r in parts[role]};parts[role]=[r for r in rows if r['id'] in ids]
    return parts


def feature_bundle(rec, split):
    path=rec['feature']/f'{split}_raw.npz'
    handoff=json.loads((rec['feature']/'handoff.json').read_text())
    if sha(path)!=handoff['artifacts'][path.name]:raise ValueError('Raw feature identity changed')
    with np.load(path,allow_pickle=False) as f:raw=f['features'];ids=f['ids'].astype(str)
    if split=='val' and ids.tolist()!=[r['id'] for r in rec['manifest']['validation']]:raise ValueError('Validation feature order mismatch')
    return raw,ids


def saved_head(name, identity, fit):
    path=OUT/'control_heads'/name;marker=path.with_suffix('.identity.json')
    if path.exists() or marker.exists():
        if not path.exists() or not marker.exists():raise ValueError('Incomplete head control; preserve it and use a new working directory')
        lock=json.loads(marker.read_text())
        if lock['identity']!=identity or lock['sha256']!=sha(path):raise ValueError('Control head identity changed')
        return ensemble.load(path,2048,15)
    model=fit();ensemble.save(path,model);write(marker,dict(identity=identity,sha256=sha(path)));return model


def fit_calibration(p,y,g):
    q=numpy_predict(dict(family='uniform'),p)[:,1];safe=np.clip(q,1e-12,1-1e-12);z=np.log(safe/(1-safe));w=np.zeros(len(y))
    for group in GROUPS:w[g==group]=1/(4*int(np.sum(g==group)))
    def objective(v):
        scores=v[0]*z+v[1];error=w*(expit(scores)-y)
        return float(np.sum(w*(np.logaddexp(0,scores)-y*scores))),np.array([error@z,error.sum()])
    result=scipy.optimize.minimize(objective,[1.,0.],jac=True,method='L-BFGS-B',bounds=[(0,None),(None,None)],
                                   options=dict(maxiter=1000,gtol=1e-10,ftol=1e-14))
    if not result.success:
        result=scipy.optimize.minimize(objective,result.x,jac=True,method='SLSQP',bounds=[(0,None),(None,None)],options=dict(maxiter=1000,ftol=1e-12))
    if not result.success or not np.isfinite(result.fun):raise ValueError('Calibration solver failed: '+str(result.message))
    thresholds=np.r_[0.,(np.unique(q)[:-1]+np.unique(q)[1:])/2,1.];best=None
    for threshold in thresholds:
        correct=(q>threshold)==y
        key=(min(float(correct[g==group].mean()) for group in GROUPS),float(correct.mean()),-abs(float(threshold)-.5),-float(threshold))
        if best is None or key>best[0]:best=(key,float(threshold))
    return dict(affine=dict(a=float(result.x[0]),b=float(result.x[1]),objective=float(result.fun),success=True),threshold=best[1])


def fit_auxiliary(rec,config):
    seed=rec['seed'];p,y,g,ids=rec['train'];identity=rec['identity'];aux={}
    if C['RUN_CALIBRATION_CONTROLS']:
        aux['calibration']=memo(f'calibration-seed{seed}.json',identity,lambda:fit_calibration(p,y,g))
    if C['RUN_SINGLE_HEAD_CONTROL']:
        one=np.repeat(p[:1],len(p),axis=0)
        def fit_single():
            fitted=[]
            for wd in config['weight_decays']:
                for family,drop in [('mlp',0.),('mlp_dropout',config['prediction_dropout'])]:
                    model,history=neural.fit(one,y,g,config,'mlp',wd,drop,10000+seed)
                    fitted.append(dict(id=f'single_head1/{family}-wd{wd}',family=family,model=model,fit=history))
            return fitted
        aux['single_head']=memo(f'single-head-seed{seed}.json',identity,fit_single)
    if not (C['RUN_DATA_ALLOCATION_CONTROLS'] or C['RUN_PARTITION_SENSITIVITY']):return aux
    raw,val_ids=feature_bundle(rec,'val');validation=rec['manifest']['validation'];lookup={name:i for i,name in enumerate(val_ids)}
    settings=rec['manifest']['training']
    if C['RUN_DATA_ALLOCATION_CONTROLS']:
        aux['full_validation']=[];labels=np.array([r['y'] for r in validation]);groups=np.array([r['group'] for r in validation])
        for epochs,label in [(20,'full_validation_20epochs'),(10,'full_validation_10epochs')]:
            recipe={**settings,'epochs':epochs};lock=identity|dict(control=label,training=recipe)
            head=saved_head(f'seed{seed}-{label}.pt',lock,lambda recipe=recipe:train_heads.train(raw,labels,groups,recipe,seed,
                dict(control=label,source_seed=seed),device=DEVICE))
            aux['full_validation'].append(dict(id=label,path=str(OUT/'control_heads'/f'seed{seed}-{label}.pt'),
                epochs=epochs,updates_per_head=(4*min(int(np.sum(groups==group)) for group in GROUPS)//recipe['batch_size'])*epochs))
    if C['RUN_PARTITION_SENSITIVITY']:
        aux['partitions']=[]
        if partition_rows(validation,0)!=rec['manifest']['parts']:raise ValueError('Independent partition algorithm does not match original partition0')
        for split_seed in C['PARTITION_SEEDS']:
            parts=partition_rows(validation,split_seed);a=parts['head_training'];b=parts['combiner_training']
            if len(a)!=720 or len(b)!=479 or {r['id'] for r in a}&{r['id'] for r in b}:raise ValueError('Partition separation failed')
            lock=identity|dict(partition_seed=split_seed,parts_hash=fingerprint(parts));name=f'seed{seed}-partition{split_seed}.pt'
            ix=[lookup[r['id']] for r in a];hy=np.array([r['y'] for r in a]);hg=np.array([r['group'] for r in a])
            head=saved_head(name,lock,lambda ix=ix,hy=hy,hg=hg:train_heads.train(raw[ix],hy,hg,settings,seed,
                dict(source_seed=seed,partition_seed=split_seed),device=DEVICE))
            bx=[lookup[r['id']] for r in b];bp=ensemble.predict(head,raw[bx],device=DEVICE)
            by=np.array([r['y'] for r in b]);bg=np.array([r['group'] for r in b])
            rows=memo(f'partition{split_seed}-seed{seed}.json',lock,lambda bp=bp,by=by,bg=bg:fit_configurations(bp,by,bg,config,seed))
            write(OUT/f'partitions/seed{seed}-split{split_seed}-manifest.json',dict(parts=parts,split_seed=split_seed,source_seed=seed))
            aux['partitions'].append(dict(split_seed=split_seed,rows=rows,head_path=str(OUT/'control_heads'/name)))
    return aux


def score_auxiliary(rec):
    p,y,g,ids=rec['test'];seed=rec['seed'];aux=rec['aux'];rows=[];partition_rows_out=[]
    uniform=numpy_predict(dict(family='uniform'),p);u=metrics(uniform,y,g)
    def add(name,prediction,**extra):
        m=metrics(prediction,y,g)
        rows.append(dict(source_seed=seed,control=name,wga=m['wga'],delta_matched_uniform_pp=100*(m['wga']-u['wga']),
                         **{f'group_{group}':m['groups'][group] for group in GROUPS},
                         **{f'count_{group}':m['counts'][group] for group in GROUPS},
                         **{f'correct_{group}':int(round(m['groups'][group]*m['counts'][group])) for group in GROUPS},**extra))
    if 'calibration' in aux:
        q=uniform[:,1];fit=aux['calibration'];safe=np.clip(q,1e-12,1-1e-12);v=expit(fit['affine']['a']*np.log(safe/(1-safe))+fit['affine']['b'])
        add('affine_uniform',np.column_stack([1-v,v]));v=(q>fit['threshold']).astype(float)
        add('training_fitted_threshold',np.column_stack([1-v,v]))
    if 'single_head' in aux:
        add('single_head1_raw',p[0]);one=np.repeat(p[:1],len(p),axis=0)
        for row in aux['single_head']:add(row['id'],numpy_predict(row,one))
    raw=None
    if 'full_validation' in aux or 'partitions' in aux:raw,test_ids=feature_bundle(rec,'test');assert test_ids.tolist()==ids.tolist()
    for control in aux.get('full_validation',[]):
        head=ensemble.load(control['path']);pred=ensemble.predict(head,raw,device=DEVICE).astype(float).mean(0)
        add(control['id'],pred,head_updates=control['updates_per_head'])
    for part in aux.get('partitions',[]):
        head=ensemble.load(part['head_path']);pred=ensemble.predict(head,raw,device=DEVICE)
        pu=metrics(pred.astype(float).mean(0),y,g)
        np.savez_compressed(OUT/f'partitions/seed{seed}-split{part["split_seed"]}-test.npz',probabilities=pred,labels=y,groups=g,ids=ids)
        for row in part['rows']:
            m=metrics(numpy_predict(row,pred),y,g)
            partition_rows_out.append(dict(source_seed=seed,partition_seed=part['split_seed'],configuration=row['id'],wga=m['wga'],
                delta_uniform_within_partition_pp=100*(m['wga']-pu['wga']),**{f'group_{group}':m['groups'][group] for group in GROUPS}))
    return rows,partition_rows_out


def source_summary(rows):
    output=[]
    groups=[('original_0_1_2',[0,1,2]),('new_3_4',[3,4]),('combined_five',[0,1,2,3,4])]
    for cohort,seeds in groups:
        for name in sorted({r['configuration'] for r in rows}):
            selected=[r for r in rows if r['configuration']==name and r['source_seed'] in seeds]
            values=[r['wga'] for r in selected];delta=[r['delta_uniform_pp'] for r in selected]
            if not selected:continue
            output.append(dict(cohort=cohort,configuration=name,n_source_seeds=len(values),mean_wga=float(np.mean(values)),
                sample_sd_wga=float(np.std(values,ddof=1)) if len(values)>1 else None,mean_delta_uniform_pp=float(np.mean(delta)),
                sample_sd_delta_pp=float(np.std(delta,ddof=1)) if len(delta)>1 else None))
    return output


def diagnostic_summaries(primary,randomness,controls,partitions):
    cohorts=[('original_0_1_2',[0,1,2]),('new_3_4',[3,4]),('combined_five',[0,1,2,3,4])]
    primary_lookup={(r['source_seed'],r['configuration']):r['wga'] for r in primary}
    summary=[];paired=[];control_summary=[];partition_summary=[]
    for cohort,seeds in cohorts:
        for wd in [.01,.001]:
            for family in ['mlp','mlp_dropout']:
                source_values=[]
                for seed in seeds:
                    rows=[r for r in randomness if r['source_seed']==seed and r['weight_decay']==wd and r['family']==family]
                    if rows:source_values.append(float(np.mean([r['wga'] for r in rows])))
                if source_values:summary.append(dict(cohort=cohort,family=family,weight_decay=wd,n_source_seeds=len(source_values),
                    initializations_per_source=len(C['COMBINER_SEEDS']),mean_wga=float(np.mean(source_values)),
                    source_sd_of_initialization_mean=float(np.std(source_values,ddof=1)) if len(source_values)>1 else None))
            for seed in seeds:
                lookup={(r['combiner_seed'],r['family']):r['wga'] for r in randomness if r['source_seed']==seed and r['weight_decay']==wd}
                deltas=[100*(lookup[(k,'mlp_dropout')]-lookup[(k,'mlp')]) for k in C['COMBINER_SEEDS'] if (k,'mlp_dropout') in lookup and (k,'mlp') in lookup]
                if deltas:paired.append(dict(cohort=cohort,source_seed=seed,weight_decay=wd,n_matched_initializations=len(deltas),
                    mean_dropout_minus_plain_pp=float(np.mean(deltas)),initialization_sd_delta_pp=float(np.std(deltas,ddof=1)) if len(deltas)>1 else None))
        for control in sorted({r['control'] for r in controls}):
            rows=[r for r in controls if r['source_seed'] in seeds and r['control']==control]
            if rows:control_summary.append(dict(cohort=cohort,control=control,n_source_seeds=len(rows),mean_wga=float(np.mean([r['wga'] for r in rows])),
                mean_delta_uniform_pp=float(np.mean([r['delta_matched_uniform_pp'] for r in rows])),
                mean_dropout_wd0_01_minus_control_pp=float(np.mean([100*(primary_lookup[(r['source_seed'],'mlp_dropout-wd0.01')]-r['wga']) for r in rows])),
                mean_dropout_wd0_001_minus_control_pp=float(np.mean([100*(primary_lookup[(r['source_seed'],'mlp_dropout-wd0.001')]-r['wga']) for r in rows]))))
        for split,configuration in sorted({(r['partition_seed'],r['configuration']) for r in partitions}):
            rows=[r for r in partitions if r['source_seed'] in seeds and r['partition_seed']==split and r['configuration']==configuration]
            if rows:partition_summary.append(dict(cohort=cohort,partition_seed=split,configuration=configuration,n_source_seeds=len(rows),
                mean_wga=float(np.mean([r['wga'] for r in rows])),mean_delta_uniform_within_partition_pp=float(np.mean([r['delta_uniform_within_partition_pp'] for r in rows]))))
    table(OUT/'combiner_randomness_summary.csv',summary);table(OUT/'combiner_paired_differences.csv',paired)
    table(OUT/'control_summary.csv',control_summary);table(OUT/'partition_summary.csv',partition_summary)


def paired_intervals(correct,groups,names,B=10000,random_seed=20261112):
    # correct[S,M,N]. Group resamples are shared across all models/source seeds.
    S,M,N=correct.shape;rng=np.random.default_rng(random_seed);draws=[];points=[]
    for group in GROUPS:
        mask=groups==group;patterns,counts=np.unique(correct[:,:,mask].reshape(S*M,-1).T,axis=0,return_counts=True)
        sampled=rng.multinomial(int(mask.sum()),counts/counts.sum(),size=B)
        draws.append((sampled@patterns.astype(float)/mask.sum()).reshape(B,S,M));points.append(correct[:,:,mask].mean(2))
    boot=np.min(draws,axis=0);point=np.min(points,axis=0)
    source_draws=rng.integers(0,S,size=(B,S));results=[]
    for wd in [.01,.001]:
        method=f'mlp_dropout-wd{wd}'
        for reference in ['uniform',f'mlp-wd{wd}']:
            i,j=names.index(method),names.index(reference);d=100*(point[:,i]-point[:,j]);b=100*(boot[:,:,i]-boot[:,:,j])
            observed=float(d.mean());sd=float(np.std(d,ddof=1)) if S>1 else None
            ci=None if S<2 else [observed-float(student_t.ppf(.975,S-1))*sd/np.sqrt(S),observed+float(student_t.ppf(.975,S-1))*sd/np.sqrt(S)]
            crossed=b[np.arange(B)[:,None],source_draws].mean(1)
            results.append(dict(method=method,reference=reference,n_source_seeds=S,mean_delta_pp=observed,per_source_delta_pp=d.tolist(),
                source_seed_t95_pp=ci,conditional_test_example_95_pp=np.percentile(b.mean(1),[2.5,97.5]).tolist(),
                crossed_source_and_test_95_pp=np.percentile(crossed,[2.5,97.5]).tolist()))
    return results


def all_uncertainty(records):
    names=['uniform','mlp-wd0.01','mlp-wd0.001','mlp_dropout-wd0.01','mlp_dropout-wd0.001'];data=[];first=None
    for rec in records:
        p,y,g,ids=rec['test'];lookup={r['id']:r for r in rec['fresh']}
        if first is None:first=(ids.copy(),y.copy(),g.copy())
        elif not all(np.array_equal(a,b) for a,b in zip((ids,y,g),first)):raise ValueError('Test IDs/labels/groups differ across source seeds')
        data.append([numpy_predict(lookup[n],p).argmax(1)==y for n in names])
    correct=np.asarray(data);output={}
    for cohort,ix in [('original_0_1_2',[0,1,2]),('new_3_4',[3,4]),('combined_five',[0,1,2,3,4])]:
        output[cohort]=paired_intervals(correct[ix],g,names,C['BOOTSTRAP_REPLICATES'],C['BOOTSTRAP_SEED'])
    write(OUT/'uncertainty.json',dict(method='paired source-seed t intervals and paired group-stratified empirical bootstrap',replicates=C['BOOTSTRAP_REPLICATES'],
        interpretation='Pointwise intervals, not simultaneous grid-wide claims. Test-example intervals condition on fitted models. Crossed intervals additionally resample source seeds; only five independent source seeds, fixed benchmark and primary split. No claim of a new test set.',cohorts=output))
    return output


def snapshot_source(rec):
    target=OUT/'source_bundles'/f'seed{rec["seed"]}'
    for name in ['experiment/manifest.json','experiment/heads.json','checkpoints/ensemble.pt','experiment/predictions/combiner_training.npz','experiment/predictions/test.npz']:
        p=rec['source']/name
        if p.is_file():
            q=target/name;q.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,q)


def checkpoint(stage,ready):
    write(OUT/'status.json',dict(status='running',stage=stage,declared_new_backbone_seeds=[3,4],completed_fits=ready,expected_source_seeds=[0,1,2,3,4]))
    archive()


def main_five():
    if C['ORIGINAL_SEEDS']!=[0,1,2] or C['ADDITIONAL_SEEDS']!=[3,4]:raise ValueError('This notebook is scoped to exactly original0/1/2 and new3/4.')
    if DEVICE!='cuda':raise RuntimeError('Enable a Kaggle GPU; this run trains two 100-epoch backbones.')
    if subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()!=COMMIT:raise RuntimeError('Wrong pinned source')
    import torchvision
    env=dict(python=platform.python_version(),torch=str(torch.__version__),torchvision=str(torchvision.__version__),numpy=np.__version__,scipy=scipy.__version__,
        pillow=PIL.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),commit=COMMIT,runner_sha256=sha(__file__))
    write(OUT/'environment.json',env)
    if C['STRICT_RUNTIME']:
        expected=dict(torch='2.11.0',torchvision='0.26.0',numpy='2.1.3',scipy='1.16.3',pillow='13.0.0',cuda='13.0')
        mismatch={k:(env[k],v) for k,v in expected.items() if str(env[k]).split('+')[0]!=v}
        if 'T4' not in env['gpu']:mismatch['gpu']=(env['gpu'],'Tesla T4')
        if mismatch:raise RuntimeError('Runtime differs from verified experiment. Preserve this report and select a matching Kaggle image. Mismatches: '+str(mismatch))
    log('Five-seed verification; train only NEW backbone seeds3,4. Existing0,1,2 reused. '+json.dumps(env))
    write(OUT/'execution_contract.json',dict(new_backbones=[3,4],reused_backbones=[0,1,2],fixed_primary_partition=0,additional_partition_seeds=C['PARTITION_SEEDS'],
        complete_configs=30,combiner_repeats_per_source=len(C['COMBINER_SEEDS']),all_fits_before_test_scoring=True,all_configs_independent=True,stage0_adapter='Only broaden factory seed guard from0/1/2 to0/1/2/3/4 in memory; original factory supplies every other recipe field; files unchanged',
        data='Waterbirds official splits',architecture='ResNet50',backbone_epochs=100,original_validation_allocation=[720,479],source_seed_counts=dict(original=3,new=2,combined=5),
        primary_contrasts='dropout MLP vs uniform and matched plain MLP, each weight decay separately; no test-based winner selection'))
    contracts()
    if C['RUN_REPOSITORY_TESTS']:
        result=subprocess.run([sys.executable,'-m','pytest','tests/methods/dpe_head_reweighting','-q','-k','not notebook','--disable-warnings'],capture_output=True,text=True,
            env={**os.environ,'PYTEST_DISABLE_PLUGIN_AUTOLOAD':'1'})
        (OUT/'repository_tests.txt').write_text(result.stdout+'\n'+result.stderr)
        if result.returncode:raise RuntimeError('Code checks failed; inspect repository_tests.txt.')
    evidence,study,plan=get_reference();config=plan['config'];records=[];dataset=get_dataset()
    for seed in [0,1,2,3,4]:
        checkpoint(f'prepare_source_seed{seed}',[r['seed'] for r in records])
        if seed<3:
            candidates=list((evidence/'head_sources').glob(f'standard-seed{seed}-*'))
            if len(candidates)!=1:raise ValueError('Ambiguous original source')
            source=candidates[0];manifest,heads,train=source_data(source)
            original=json.loads((study/f'comparisons/seed{seed}/models.json').read_text());write(OUT/f'original_models/seed{seed}.json',original)
            for name,expected in original['lock']['inputs'].items():
                if sha(source/name)!=expected:raise ValueError('Original input identity mismatch')
            feature=next((evidence/'features/standard').glob(f'seed{seed}-*'));backbone=evidence/f'backbones/standard/seed{seed}/ckpt_last.pt';handoff=None
            log(f'Reusing original backbone/head source seed{seed}; no backbone training')
        else:
            # The only backbone-training call: this branch is reachable for3,4 only.
            log(f'Training/resuming NEW backbone seed{seed}: 100 epochs, unchanged recipe')
            source,handoff=sources.build_source(W/'additional_sources',dataset,'standard',seed,DEVICE,C['WORKERS'],True)
            manifest,heads,train=source_data(source);original=None
            feature=Path(handoff['test_features']).parent;backbone=Path(handoff['backbone'])
        identity=dict(source_seed=seed,manifest=sha(source/'experiment/manifest.json'),heads=sha(source/'checkpoints/ensemble.pt'),
            training_predictions=sha(source/'experiment/predictions/combiner_training.npz'),config=fingerprint(config),runtime=fingerprint(env))
        p,y,g,ids=train
        fresh=memo(f'primary-seed{seed}.json',identity,lambda p=p,y=y,g=g,seed=seed:fit_configurations(p,y,g,config,seed))
        rec=dict(seed=seed,source=source,manifest=manifest,heads=heads,train=train,original_rows=original['candidates'] if original else fresh,
            fresh=fresh,feature=feature,checkpoint=backbone,handoff=handoff,identity=identity)
        rec['aux']=fit_auxiliary(rec,config)
        write(OUT/f'auxiliary/seed{seed}.json',rec['aux'])
        if C['RUN_COMBINER_STABILITY']:
            def fit_randomness(rec=rec,p=p,y=y,g=g,seed=seed):
                rows=[]
                for random_seed in C['COMBINER_SEEDS']:
                    for wd in config['weight_decays']:
                        for family,drop in [('mlp',0.),('mlp_dropout',config['prediction_dropout'])]:
                            model,fit=neural.fit(p,y,g,config,'mlp',wd,drop,random_seed)
                            rows.append(dict(source_seed=seed,combiner_seed=random_seed,weight_decay=wd,family=family,model=model,fit=fit))
                return rows
            rec['randomness']=memo(f'randomness-seed{seed}.json',identity|dict(combiner_seeds=C['COMBINER_SEEDS']),fit_randomness)
        else:rec['randomness']=[]
        records.append(rec);snapshot_source(rec);checkpoint(f'all_fits_frozen_seed{seed}',[r['seed'] for r in records]);log(f'All requested fits frozen for source seed{seed}')
    if [r['seed'] for r in records]!=[0,1,2,3,4]:raise ValueError('Cannot report a five-seed result with a missing source')
    checkpoint('all_five_sources_frozen_before_test',[0,1,2,3,4]);log('All five sources and controls frozen. Starting test scoring.')
    comparison=[];randomness=[];controls=[];partitions=[]
    for rec in records:
        seed=rec['seed'];source=rec['source']
        if rec['handoff'] is not None:sources.ensure_test_cache(rec['handoff'])
        if seed<3:
            reference=json.loads((study/f'comparisons/seed{seed}/metrics.json').read_text())
            if sha(source/'experiment/predictions/test.npz')!=reference['test_cache_hash']:raise ValueError('Reference test cache changed')
            if sha(study/f'comparisons/seed{seed}/models.json')!=reference['models_hash']:raise ValueError('Reference model record changed')
        p,y,g,ids=sources.load_cache(source,'test',rec['manifest'],rec['heads']['head_ids'],rec['heads']['lock']['ensemble']);rec['test']=(p,y,g,ids)
        if len(y)!=5794 or [int(np.sum(g==group)) for group in GROUPS]!=[2255,2255,642,642]:raise ValueError('Wrong official test counts')
        lookup={r['id']:r for r in rec['original_rows']};u=metrics(numpy_predict(lookup['uniform'],p),y,g)
        for row in rec['fresh']:
            predicted,_=neural.candidate_prediction(row,p);independent=numpy_predict(row,p);np.testing.assert_allclose(predicted,independent,atol=1e-12,rtol=0)
            m=metrics(predicted,y,g);old=numpy_predict(lookup[row['id']],p)
            if seed<3:
                old_metric=metrics(old,y,g);expected=reference['results'][row['id']]['metrics']
                changed=int((old.argmax(1)!=predicted.argmax(1)).sum())
                if changed:
                    write(OUT/f'mismatch-seed{seed}-{row["id"]}.json',dict(configuration=row['id'],changed_decisions=changed))
                    raise ValueError('Original source decisions changed; investigate before combining five seeds')
                if abs(old_metric['wga']-expected['wga'])>1e-14:raise ValueError('Original WGA does not reproduce')
            comparison.append(dict(source_seed=seed,cohort='original' if seed<3 else 'new',configuration=row['id'],wga=m['wga'],delta_uniform_pp=100*(m['wga']-u['wga']),
                changed_reference_decisions=int((old.argmax(1)!=predicted.argmax(1)).sum()) if seed<3 else None,
                **{f'group_{group}':m['groups'][group] for group in GROUPS},**{f'count_{group}':m['counts'][group] for group in GROUPS},
                **{f'correct_{group}':int(round(m['groups'][group]*m['counts'][group])) for group in GROUPS}))
        for row in rec['randomness']:
            predicted,_=neural.predict(row['model'],p);m=metrics(predicted,y,g)
            randomness.append({k:row[k] for k in ['source_seed','combiner_seed','weight_decay','family']}|dict(wga=m['wga'],delta_uniform_pp=100*(m['wga']-u['wga'])))
        control_rows,partition_rows_=score_auxiliary(rec);controls+=control_rows;partitions+=partition_rows_
        table(OUT/'per_seed.csv',comparison);table(OUT/'combiner_randomness.csv',randomness)
        # Controls have heterogeneous metadata: JSON is canonical, CSV uses all keys.
        write(OUT/'controls.json',controls)
        if controls:table(OUT/'controls.csv',[{k:r.get(k) for k in sorted({key for x in controls for key in x})} for r in controls])
        table(OUT/'partition_sensitivity.csv',partitions)
        if C['RUN_IMAGE_REPLAY']:
            replay=image_replay(seed,source,rec['checkpoint'],rec['feature'],dataset,rec['fresh'],p,y,g)
            if any(r['changed_decisions'] for r in replay['configurations'].values()):raise ValueError('Image replay changed class decisions; retain outputs for investigation')
        snapshot_source(rec);write(OUT/'status.json',dict(status='running',stage='scoring',source_seeds=[0,1,2,3,4],scored_seeds=sorted({r['source_seed'] for r in comparison})))
        archive();log(f'Source seed{seed} scored and archived')
    summary=source_summary(comparison);table(OUT/'summary_by_cohort.csv',summary)
    diagnostic_summaries(comparison,randomness,controls,partitions)
    uncertainty=all_uncertainty(records) if C['RUN_TEST_BOOTSTRAP'] else None
    lines=['# Five-seed DPE aggregation verification','','Only backbone seeds3 and4 were newly trained. Original0/1/2 were reused.',
        'Waterbirds, ResNet50, primary partition0 and all training settings are unchanged. Every configuration is independently reported.',
        'Accuracy is in percent below; CSV accuracy is fractional and differences are percentage points.',
        'Original3, new2 and combined5 are reported separately. No extra candidate-selection split is introduced.',
        '', '| Cohort | Configuration | Source seeds | Mean test WGA (%) | SD (pp) | Change vs uniform (pp) |',
        '| --- | --- | ---: | ---: | ---: | ---: |']
    for r in summary:
        if r['configuration'] in ['uniform','logistic-wd0.01','logistic-wd0.001','mlp-wd0.01','mlp-wd0.001','mlp_dropout-wd0.01','mlp_dropout-wd0.001']:
            lines.append(f"| {r['cohort']} | {r['configuration']} | {r['n_source_seeds']} | {100*r['mean_wga']:.4f} | {100*r['sample_sd_wga']:.4f} | {r['mean_delta_uniform_pp']:+.4f} |")
    lines+=['','## How to assess the evidence','',
        'Compare dropout MLP with both uniform and matched plain MLP; do not select the best weight decay after seeing test scores.',
        'Use source-seed paired differences and uncertainty.json. Conditional test-example intervals alone do not capture training variation.',
        'control_summary.csv, partition_summary.csv and combiner_randomness_summary.csv summarize the diagnostics by cohort; combiner_paired_differences.csv pairs plain/dropout initializations within each source.',
        'Combiner repetitions and alternative partitions are nested within five backbones; never count them as additional independent backbone seeds.',
        'Inspect new3/4 separately from the previously explored0/1/2. Five seeds still give limited precision; an inconclusive or negative result must be retained.',
        '', '## Mechanism and fairness controls','',
        'controls.csv includes positive-slope affine calibration, training-fitted thresholds, replicated fixed head1, and full-validation uniform heads.',
        'The full-validation20-epoch control has40 updates/head; the10-epoch version has20, matching the primary update count. Selection frequency/pools still differ, so neither is a pure single-factor causal test.',
        'partition_sensitivity.csv reports two additional720/479 partitions on the SAME five backbones, with all30 procedures on each partition.',
        'These checks assess WGA, calibration explanations, prediction-dropout value, data allocation and split sensitivity. They do not establish a background-causality or cross-dataset claim.',
        '', 'Portable head/prediction bundles are in source_bundles/. Large backbones/raw features/images remain in the full Kaggle output for resume.',
        'Checkpointing is per epoch during new backbone training; reports/compact ZIP are refreshed after each source is frozen/scored.']
    (OUT/'report.md').write_text('\n'.join(lines)+'\n')
    write(OUT/'status.json',dict(status='complete',mode='five_seed_verification',new_backbone_seeds=[3,4],reused_source_seeds=[0,1,2],source_seeds=[0,1,2,3,4],
        source_seed_count=5,primary_rows=len(comparison),combiner_randomness_rows=len(randomness),partition_rows=len(partitions),controls=len(controls)))
    log('Completed five-seed verification. Download '+str(archive()))


def budget_expired(signum,frame):
    raise TimeoutError('Declared wall-time budget reached. Preserve the full Kaggle output to resume from the last completed epoch.')


signal.signal(signal.SIGALRM,budget_expired)
signal.setitimer(signal.ITIMER_REAL,C['TIME_BUDGET_HOURS']*3600)
try:
    main_five()
except BaseException as exc:
    previous=json.loads((OUT/'status.json').read_text()) if (OUT/'status.json').exists() else {}
    write(OUT/'status.json',dict(status='incomplete' if isinstance(exc,(KeyboardInterrupt,TimeoutError)) else 'failed',exception=type(exc).__name__,message=str(exc),last_progress=previous,
        expected_source_seeds=[0,1,2,3,4],do_not_report_five_seed_result=True))
    archive()
    raise
finally:
    signal.setitimer(signal.ITIMER_REAL,0)
'''
runner_path = work / 'five_seed_verification_runner.py'
runner_path.write_text(RUNNER_SOURCE)
print('Worker written:', runner_path)


## Run both new seeds and all checks on Kaggle

The worker uses your Kaggle resources and GPU allowance. No experiments are launched in Codex by this notebook. Keep the session active or use Save Version / Run All as appropriate for your Kaggle workflow.

Logs identify reused versus newly trained sources. A time-budget interruption is an incomplete run, not a scientific failure; preserve the full output for resume. A numerical/data/reproduction failure must be inspected rather than hidden or replaced by a better test result. To preserve partial results after an error, run the final download cell manually.


In [ ]:
import signal

logs = work / "logs"
logs.mkdir(exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(runtime) + os.pathsep + str(repo) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["OMP_NUM_THREADS"] = str(CPU_THREADS)
env["MKL_NUM_THREADS"] = str(CPU_THREADS)
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
log_path = logs / (MODE + ".log")
with log_path.open("w", buffering=1) as logfile:
    process = subprocess.Popen([sys.executable, "-u", str(runner_path), str(config_path)],
                               cwd=repo, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait()
        raise
if returncode:
    raise RuntimeError(f"Verification stopped with exit code {returncode}. Inspect {log_path}. Run the download cell below to preserve any partial results.")
print("Finished. Download the results below.")


## Download the five-seed result bundle

Share **`verification_results.zip`** when complete. Main files:

- `status.json`: requires both new seeds3/4 and total source count5 for completion.
- `per_seed.csv`:150 main configuration/seed results with per-group correct counts and WGA.
- `summary_by_cohort.csv`: original0/1/2, new3/4 and combined0–4 kept separate.
- `uncertainty.json`: paired differences and clearly separated source-seed/test-example uncertainty for each cohort. Five source seeds still provide limited precision; intervals are pointwise, not simultaneous claims across the entire grid.
- `combiner_randomness.csv`:200 fits nested within five backbones; compare dropout with plain MLP at each weight decay.
- `control_summary.csv`, `partition_summary.csv`, `combiner_randomness_summary.csv` and `combiner_paired_differences.csv`: cohort summaries and within-source paired comparisons.
- `controls.csv` and `controls.json`: calibration, threshold, fixed-head and full-validation data/budget controls.
- `partition_sensitivity.csv`:300 alternative-partition configuration results nested within the same five backbones.
- `images/`: raw-image reproduction checks; any changed decisions stop a clean-completion verdict.
- `source_bundles/`, `frozen/`, `control_heads/`: portable head/prediction evidence and fitted states, plus input identities.

Report every declared outcome, including negative or inconclusive comparisons. Do not choose a weight decay, partition or seed by test performance. A positive comparison with uniform does not by itself establish a dropout-specific benefit, reduced background reliance, cross-dataset generality or novelty.

The compact ZIP includes portable head/model evidence and reports. **Large backbone/optimizer checkpoints, raw feature arrays and images stay in the full notebook output**; retain it for recovery.


In [ ]:
from IPython.display import FileLink, Markdown, display

result_zip = work / "verification_results.zip"
if result_zip.is_file():
    display(FileLink(str(result_zip), result_html_prefix="Download compact verification results: "))
else:
    print("No archive exists yet. Check the setup/worker log and preserve the working directory.")
for report in sorted((work / "results").glob("*/report.md")):
    display(Markdown(f"### {report.parent.name}"))
    display(Markdown(report.read_text()))
print("For backbone resume, preserve the full notebook output, not only the compact ZIP.")
